# Train OpsiGen on a Custom Opsin Dataset

This notebook demonstrates the WDS/custom opsin workflow using Python imports rather than shell commands. It prepares a sequence plus lambda-max metadata dataset, configures bovine-numbered spectral tuning sites as graph nodes, runs preprocessing from FASTA/PDB structures, launches training, and inspects outputs.

![OpsiGen overview](Images/main_figure.png)


## 1. Imports and editable paths

Set `WDS_META`, `WDS_FASTA`, and `PDB_DIR` for your machine. `WDS_FASTA` should be the unaligned sequence FASTA. The PDB directory should contain one `.pdb` per sequence ID; matching `.json` pLDDT files are recorded for traceability but ignored during preprocessing and training.


In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import shutil

import pandas as pd

from opsigen.config import FunctionalTrainingConfig, PreprocessRunConfig, TrainingConfig, PredictionConfig
from opsigen.dataset_prep import prepare_wds_dataset
from opsigen.preprocessing import preprocess_opsins
from opsigen.functional_training import train_functional_classifier
from opsigen.training import train_model
from opsigen.prediction import run_predictions

REPO_ROOT = Path.cwd()

WDS_META = Path('/Users/frazer/Desktop/wds_meta.tsv')
WDS_FASTA = Path('/Users/frazer/Desktop/wds.fasta')
PDB_DIR = None  # Example: Path('/path/to/folder_with_pdb_and_plddt_json_files')

DATASET_DIR = REPO_ROOT / 'datasets' / 'wds'
CONFIG_DIR = REPO_ROOT / 'configs'

print('Repository:', REPO_ROOT)
print('Metadata exists:', WDS_META.exists(), WDS_META)
print('FASTA exists:', WDS_FASTA.exists(), WDS_FASTA)
print('PDB directory:', PDB_DIR)


## 2. Choose bovine-numbered spectral tuning sites

Hagen et al. use bovine rhodopsin sequence positions as the numbering convention. The broad Figure 3 set is useful when your dataset mixes animal opsin families; subtype-specific sets are included for narrower experiments.


In [ ]:
SITE_PRESETS = {
    'hagen_figure3_broad': [46, 49, 52, 83, 86, 90, 93, 118, 122, 124, 132, 180, 197, 230, 233, 277, 285, 292, 298, 299, 300, 308],
    'rh1_example': [83, 122, 292, 299, 300],
    'sws1_example': [86, 90, 93, 118],
    'sws1_mammal_text': [46, 49, 50, 52, 86, 90, 93, 114, 118],
    'lws_mws_five_site_rule': [180, 197, 277, 285, 308],
}

SELECTED_SITE_SET = 'hagen_figure3_broad'
REFERENCE_RESIDUE_SITES = SITE_PRESETS[SELECTED_SITE_SET]

pd.DataFrame({
    'preset': list(SITE_PRESETS),
    'site_count': [len(v) for v in SITE_PRESETS.values()],
    'sites': [','.join(map(str, v)) for v in SITE_PRESETS.values()],
})


## 3. Prepare the dataset and configs

This writes the training table, one FASTA per sequence, train/test split files, preprocessing manifest, and WDS-specific configs. If `PDB_DIR` is not set yet, the manifest will have empty `pdb_path` values; set `PDB_DIR` and rerun this cell before preprocessing.


In [ ]:
prepared = prepare_wds_dataset(
    meta_path=WDS_META,
    fasta_path=WDS_FASTA,
    pdb_dir=PDB_DIR,
    output_dir=DATASET_DIR,
    configs_dir=CONFIG_DIR,
    reference_sequence_id='Bovine',
    reference_residue_sites=REFERENCE_RESIDUE_SITES,
)

print('Training table:', prepared.training_excel)
print('Preprocess config:', prepared.preprocess_config)
print('Training config:', prepared.train_config)
print('Functional classifier config:', prepared.functional_train_config)
print('Prediction config:', prepared.predict_config)
print('Validation report:', prepared.validation_report)


## 4. Inspect the prepared training table

The model trains from `lmax` and the graph paths that preprocessing will write under `runs/preprocess/wds`.


In [ ]:
training_table = pd.read_excel(prepared.training_excel) if prepared.training_excel.suffix == '.xlsx' else pd.read_csv(prepared.training_csv)
validation_report = pd.read_csv(prepared.validation_report)
manifest = pd.read_csv(prepared.preprocess_manifest)

summary = {
    'training_rows': len(training_table),
    'unique_sequences': training_table['Name'].nunique(),
    'lambda_missing': int(training_table['lmax'].isna().sum()),
    'pdb_paths_filled': int(manifest['pdb_path'].fillna('').astype(str).str.strip().ne('').sum()),
    'rows_with_non_standard_residues': int(validation_report['non_standard_residues'].fillna('').astype(str).str.strip().ne('').sum()),
}

print(json.dumps(summary, indent=2))
display(training_table[['Name', 'Wildtype', 'lmax', 'Opsin_Family', 'features_path', 'dists_path', 'pdb_path']].head())
display(validation_report[validation_report['non_standard_residues'].fillna('').astype(str).str.strip().ne('')].head(10))


## 5. Inspect preprocessing configuration

The key fields are `reference_alignment`, `reference_alignment_input`, `reference_sequence_id`, and `reference_residue_sites`. The WDS FASTA is unaligned input, but residue-site mapping happens through a shared animal-opsin MSA so selected nodes represent homologous bovine-numbered positions.


In [ ]:
preprocess_config = PreprocessRunConfig.from_file(prepared.preprocess_config)
pp = preprocess_config.preprocessing

print('Reference alignment:', pp.reference_alignment)
print('Reference alignment input:', pp.reference_alignment_input)
print('Reference sequence:', pp.reference_sequence_id, pp.reference_sequence_path)
print('Selected site count:', len(pp.reference_residue_sites))
print('Selected sites:', pp.reference_residue_sites)
print('Gap strategy:', pp.site_gap_strategy)
print('Output directory:', pp.output_dir)


## 6. Check preprocessing readiness

For the WDS/bovine-reference workflow, preprocessing requires MAFFT to build the shared animal-opsin MSA unless `reference_alignment` already exists. It also requires `feature_maker/interface2grid`, `feature_maker/chem.lib`, the amino-acid mapping file, and populated PDB paths.


In [ ]:
mafft_available = shutil.which(pp.mafft_executable) is not None or Path(pp.mafft_executable).exists()
alignment_exists = pp.reference_alignment.exists()
alignment_input_exists = pp.reference_alignment_input is not None and pp.reference_alignment_input.exists()

readiness_rows = [
    {'check': 'reference MSA exists', 'ok': alignment_exists, 'value': str(pp.reference_alignment)},
    {'check': 'reference MSA input', 'ok': alignment_exists or alignment_input_exists, 'value': str(pp.reference_alignment_input)},
    {'check': 'MAFFT available if MSA must be built', 'ok': alignment_exists or mafft_available, 'value': pp.mafft_executable},
    {'check': 'feature maker binary', 'ok': pp.feature_maker_binary.exists(), 'value': str(pp.feature_maker_binary)},
    {'check': 'chem.lib', 'ok': pp.chem_lib_path.exists(), 'value': str(pp.chem_lib_path)},
    {'check': 'amino mapping', 'ok': pp.amino_mapping_path.exists(), 'value': str(pp.amino_mapping_path)},
    {'check': 'reference sequence FASTA', 'ok': pp.reference_sequence_path.exists(), 'value': str(pp.reference_sequence_path)},
    {'check': 'all manifest PDB paths populated', 'ok': manifest['pdb_path'].fillna('').astype(str).str.strip().ne('').all(), 'value': f"{manifest['pdb_path'].fillna('').astype(str).str.strip().ne('').sum()}/{len(manifest)}"},
]

readiness = pd.DataFrame(readiness_rows)
display(readiness)


## 7. Run preprocessing

This can take a while because it builds or reuses the reference MSA, maps bovine-numbered sites, cuts residues from each PDB, runs the native feature maker, appends amino-acid descriptors, and writes distance matrices. Leave the toggle off until readiness checks are green.


In [ ]:
RUN_PREPROCESSING = False

if RUN_PREPROCESSING:
    preprocessed = preprocess_opsins(preprocess_config.records, preprocess_config.preprocessing)
    preprocessed_table = pd.DataFrame([record.__dict__ for record in preprocessed])
    display(preprocessed_table.head())
else:
    print('Set RUN_PREPROCESSING = True after setting PDB_DIR and passing readiness checks.')


## 8. Load and optionally shorten the training run

For a demo, set `DEMO_EPOCHS` to a small value. For a real run, use the value in `configs/train.wds.json` or edit that config directly.

`data.indexes_to_keep` selects feature columns from each selected node. It is not the list of bovine residue sites; those graph nodes were chosen earlier by `preprocessing.reference_residue_sites`.


In [ ]:
training_config = TrainingConfig.from_file(prepared.train_config)

DEMO_EPOCHS = 2
training_config = replace(training_config, fit=replace(training_config.fit, epochs=DEMO_EPOCHS))

print('Training table:', training_config.data.excel_path)
print('Feature column:', training_config.data.features_column)
print('Distance column:', training_config.data.dists_column)
print('Selected graph-node sites:', len(pp.reference_residue_sites), pp.reference_residue_sites)
print('Feature columns kept:', len(training_config.data.indexes_to_keep), training_config.data.indexes_to_keep)
print('Model input feature count:', training_config.model.number_features)
print('Target column:', training_config.data.target_column)
print('Epochs for this notebook run:', training_config.fit.epochs)
print('Output directory:', training_config.output_dir)


## 9. Train the model

Training expects graph files to exist at the paths in `wds_training.xlsx`, which are produced by preprocessing.


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    training_result = train_model(training_config)
    print(training_result)
else:
    print('Set RUN_TRAINING = True after preprocessing has produced graph feature and distance files.')


## 10. Inspect training outputs

Training writes checkpoints, normalization arrays, metrics, and run metadata.


In [ ]:
train_output = training_config.output_dir
metrics_path = train_output / 'logs' / 'metrics.jsonl'
metadata_path = train_output / 'metadata.json'

print('Expected final checkpoint:', train_output / 'checkpoints' / training_config.fit.checkpoint_name)
print('Expected means:', train_output / 'artifacts' / 'means.npy')
print('Expected stds:', train_output / 'artifacts' / 'stds.npy')

if metrics_path.exists():
    metrics = pd.read_json(metrics_path, lines=True)
    display(metrics.tail())
else:
    print('No metrics file found yet:', metrics_path)

if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text())
    print(json.dumps(metadata.get('outputs', {}), indent=2))
else:
    print('No training metadata found yet:', metadata_path)


## 11. Train a functional classifier

The wavelength regressor skips `lmax = 0` rows. This classifier keeps those rows and treats `lmax <= 0` as non-functional by default. Because non-functional examples are usually rare, the config uses minority oversampling and class-weighted binary cross-entropy.


In [ ]:
functional_config = FunctionalTrainingConfig.from_file(prepared.functional_train_config)
functional_config = replace(functional_config, fit=replace(functional_config.fit, epochs=DEMO_EPOCHS))

functional_labels = training_table['lmax'].astype(float) > functional_config.classifier.functional_threshold_nm
display(functional_labels.map({True: 'functional', False: 'nonfunctional'}).value_counts().rename_axis('class').reset_index(name='count'))

print('Positive class:', functional_config.classifier.positive_class)
print('Functional threshold nm:', functional_config.classifier.functional_threshold_nm)
print('Class-weighted loss:', functional_config.classifier.class_weighted_loss)
print('Weighted sampler:', functional_config.fit.weighted_sampler)
print('Monitor metric:', functional_config.classifier.monitor_metric)
print('Output directory:', functional_config.output_dir)


In [ ]:
RUN_FUNCTIONAL_TRAINING = False

if RUN_FUNCTIONAL_TRAINING:
    functional_result = train_functional_classifier(functional_config)
    print(functional_result)
else:
    print('Set RUN_FUNCTIONAL_TRAINING = True after preprocessing has produced graph feature and distance files.')


In [ ]:
functional_metrics_path = functional_config.output_dir / 'logs' / 'metrics.jsonl'
functional_metadata_path = functional_config.output_dir / 'metadata.json'

print('Expected classifier checkpoint:', functional_config.output_dir / 'checkpoints' / functional_config.fit.checkpoint_name)
print('Expected classifier predictions:', functional_config.output_dir / 'validation_predictions.csv')

if functional_metrics_path.exists():
    functional_metrics = pd.read_json(functional_metrics_path, lines=True)
    display(functional_metrics[['epoch', 'test_balanced_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_average_precision']].tail())
else:
    print('No classifier metrics file found yet:', functional_metrics_path)

if functional_metadata_path.exists():
    functional_metadata = json.loads(functional_metadata_path.read_text())
    print(json.dumps({k: functional_metadata.get(k) for k in ['positive_class', 'train_class_counts', 'test_class_counts', 'best_metric']}, indent=2))
else:
    print('No classifier metadata found yet:', functional_metadata_path)


## 12. Configure batch prediction from the trained WDS model

The generated prediction config points to the WDS checkpoint and normalization arrays. Replace `datasets/wds/predict_manifest.template.csv` with real new opsin FASTA/PDB rows before running prediction.


In [ ]:
prediction_config = PredictionConfig.from_file(prepared.predict_config)

print('Prediction model:', prediction_config.model_path)
print('Means:', prediction_config.means_path)
print('Stds:', prediction_config.stds_path)
print('Prediction output:', prediction_config.output_dir)
print('Prediction records:', len(prediction_config.records))
for record in prediction_config.records[:3]:
    print(record)


## 13. Run prediction

Prediction can preprocess new FASTA/PDB records first, then run inference with the trained WDS model.


In [ ]:
RUN_PREDICTION = False

if RUN_PREDICTION:
    prediction_result = run_predictions(prediction_config)
    predictions = pd.read_csv(prediction_result.predictions_path)
    display(predictions)
else:
    print('Set RUN_PREDICTION = True after training a WDS model and filling the prediction manifest.')
